In [1]:
import pathlib

import numpy as np
import pandas as pd
import pyarrow as pa

from pyarrow import parquet as pq
from tqdm.auto import tqdm

In [2]:
output_file = "species_dlpno_results.parquet"

In [3]:
cwd = pathlib.Path().absolute()
quantum_green = pathlib.Path("/home/shared/projects/quantum_green")
paquets = quantum_green / "supercloud_xfer"
data_dir = cwd.parent.parent / "data" / "qm_results"

In [4]:
pd.set_option("display.max_columns", None)


def head(df, n=2):
    display(df.head(n))
    print(f"Contains {len(df)} rows")


def swifter_apply(seris, func, desc="Applying function"):
    return seris.swifter.progress_bar(True, desc=desc).apply(func)


def extract_last(lst):
    if isinstance(lst, list) and len(lst) > 0:
        return lst[-1]
    return lst

In [5]:
species_data = pd.read_pickle(
    quantum_green / "reactants" / "quantum_green_species_data_24march12b.pkl"
)
head(species_data)

,asmi,species_id,species_dft_log_path,species_dft_route_section,charge,multiplicity,species_dft_opt_max_steps,species_dft_opt_normal_termination_check,species_dft_opt_cpu_time,species_dft_opt_wall_time,species_dft_sum_of_electronic_and_thermal_enthalpies_hartree,species_dft_hartreefock_energy_hartree,species_dft_zpe_hartree,species_dft_e0_zpe_hartree,species_dft_gibbs_hartree,species_dft_scf_hartree,species_dft_frequency_modes,species_dft_std_forces,species_dft_opted_std_xyz,species_dft_opted_xyz_input_orientation,is_ts,batch_label,species_dft_log_filename_id,nasmi,matched_rxn_fingerprint,species_dft_log_source_utf_8_sha512,passed_connectivity_check,rdkit_canonical_nasmi,partent_rdkit_canonical_nasmi_radical_only,reaction_center_atom_map_num_radical_only,std_xyz_str,xyz_str,matched_parent_mol_sha512_radical_only,matched_parent_mol_asmi_radical_only,matched_parent_mol_reaction_center_atom_map_num_radical_only,species_dft_first_freq,species_dlpno_log_filename_id,species_dlpno_log_path,species_dlpno_sp_hartree,dlpno_batch_label,species_dft_frequencies
0,[C:1]([C:2]1([H:13])[C:3]([H:14])([H:15])[N:4]...,1,/home/gridsan/groups/RMG/Projects/Hao-Wei-Osca...,"P opt=(calcfc,maxcycle=128,noeig,nomicro,carte...",0,2,102,True,4100.1,1076.5,-434.175602,-434.321489,0.136748,-434.184741,-434.218451,434.321490,"[[[1.0, 6.0, -0.01, 0.18, 0.17], [2.0, 6.0, -0...","[[1.0, 6.0, 1.274e-05, 2.05e-06, 3.763e-06], [...","[[1.0, 6.0, 0.0, 1.687648, -1.481461, 0.769379...","[[1.0, 6.0, 0.0, 1.711519, -1.472917, 0.740063...",False,aug11b,id52810,CC1C[N]C(=N)N1C=O,"{('46e86ef9080f8c5b21f5c329d23f0d6c', 'p2_smi')}",42d67c486eb53d22486a5d1da768e644bf644469198894...,True,CC1C[N]C(=N)N1C=O,CC1CNC(=N)N1C=O,4.0,17\n\nC 1.687648 -1.481461 0.769379\nC 1.14635...,17\n\nC 1.711519 -1.472917 0.740063\nC 1.17193...,4f388644fa3d52bec962545fb1a65c4446aa8758d8834c...,[C:1]([C:2]1([H:13])[C:3]([H:14])([H:15])[N:4]...,16,69.0965,id52810,/home/gridsan/jburns/RMG_shared/Projects/Hao-W...,-434.224944,reactant_aug11b,"[69.0965, 133.1139, 185.0048, 247.5052, 269.23..."
1,[C:1]([C:2]1([H:13])[C:3]([H:14])([H:15])[O:4]...,3,/home/gridsan/groups/RMG/Projects/Hao-Wei-Osca...,"P opt=(calcfc,maxcycle=128,noeig,nomicro,carte...",0,2,126,True,12128.7,3218.9,-403.235873,-403.429122,0.183595,-403.245527,-403.280167,403.429122,"[[[1.0, 6.0, 0.21, 0.06, 0.18], [2.0, 6.0, 0.0...","[[1.0, 6.0, 5.513e-06, 2.7033e-05, 2.003e-06],...","[[1.0, 6.0, 0.0, -2.800669, 0.461226, 0.774726...","[[1.0, 6.0, 0.0, 2.701051, 1.034704, 0.04014],...",False,aug11b,id52729,CC1COC(C2C[N]2)C1,"{('5ddcee1ad82096fe7dcfd17162f30b09', 'p2_smi')}",7667c6f680d9065479d515666e4c3246f65704e4926140...,True,CC1COC(C2C[N]2)C1,CC1COC(C2CN2)C1,8.0,21\n\nC -2.800669 0.461226 0.774726\nC -1.7902...,21\n\nC 2.701051 1.034704 0.04014\nC 1.73068 -...,f0d1bccdf82215b9726f7ea9b5b912db1d1b0a4b8fd7e0...,[C:1]([C:2]1([H:13])[C:3]([H:14])([H:15])[O:4]...,20,40.2091,id52729,/home/gridsan/jburns/RMG_shared/Projects/Hao-W...,-403.298238,reactant_aug11b,"[40.2091, 100.6031, 169.247, 232.8355, 250.021..."


Contains 348258 rows


In [6]:
matching_df = pd.read_csv(cwd / "quantum_green_species_data_match.csv")
matching_df = (
    matching_df[matching_df["dlpno_index"] >= 0]
    .sort_values("dlpno_index")
    .reset_index(drop=True)
    .copy()
)
matching_df["smiles"] = matching_df["species_id"].map(
    species_data.set_index("species_id")["asmi"]
)
head(matching_df)

,species_id,dft_index,dlpno_index,semiempirical_index,rmsd,from_initial,smiles
0,187743,187762,0,1062246,0.0,True,[C:1]([C:2]([C:3]1([H:18])[O:4][C:5]([C:6]([H:...
1,186835,187180,1,1067416,0.0,False,[C:1](=[C:2]([C:3]([C:4]([H:15])([H:16])[H:17]...


Contains 340698 rows


In [7]:
def read_column(name):
    return pq.read_table(paquets / "dlpno_nonts_dlpno_v6.parquet", columns=[name])[name]

In [8]:
energy = read_column("energy")
keep_index = np.zeros(len(energy), dtype=bool)
keep_index[matching_df["dlpno_index"].values] = True
row_filter = pa.array(keep_index)
sum(keep_index).item(), row_filter.sum().as_py()

(340698, 340698)

In [9]:
column_metadata = {
    "smiles": "Reaction SMILES (with atom mapping indicating xyz and force order)",
    "route_section": "Level of theory",
    "charge": "Molecular formal charge",
    "multiplicity": "Electron multiplicity",
    "energy": "Final single point energy",
    "run_time": "Total run time",
    "input_coordinates": "Input xyz coords",
    "dipole_au": "Molecular dipole in Atomic Units (AU)",
    "t1_diagnostic": "T1 diagnostic",
}

In [10]:
dlpno_data = pa.Table.from_arrays(
    [pa.array(matching_df["smiles"])],
    schema=pa.schema([pa.field("smiles", pa.string())], metadata=column_metadata),
)

pbar = tqdm([key for key in column_metadata.keys() if key != "smiles"])
for name in pbar:
    pbar.set_description(f"Processing column {name}")
    column = read_column(name).filter(row_filter)
    if name in ["scf", "std_xyz", "std_forces"]:
        column = pa.array([extract_last(item.as_py()) for item in column])
    old_dlpno_data = dlpno_data
    dlpno_data = old_dlpno_data.append_column(name, column)
    del old_dlpno_data
    del column

  0%|          | 0/8 [00:00<?, ?it/s]

In [11]:
dlpno_data.schema

smiles: string
route_section: string
charge: uint8
multiplicity: uint8
energy: double
run_time: uint32
input_coordinates: list<element: list<element: double>>
  child 0, element: list<element: double>
      child 0, element: double
dipole_au: float
t1_diagnostic: float
-- schema metadata --
smiles: 'Reaction SMILES (with atom mapping indicating xyz and force orde' + 2
route_section: 'Level of theory'
charge: 'Molecular formal charge'
multiplicity: 'Electron multiplicity'
energy: 'Final single point energy'
run_time: 'Total run time'
input_coordinates: 'Input xyz coords'
dipole_au: 'Molecular dipole in Atomic Units (AU)'
t1_diagnostic: 'T1 diagnostic'

In [12]:
dlpno_df = dlpno_data.to_pandas()
head(dlpno_df)

,smiles,route_section,charge,multiplicity,energy,run_time,input_coordinates,dipole_au,t1_diagnostic
0,[C:1]([C:2]([C:3]1([H:18])[O:4][C:5]([C:6]([H:...,uHF UNO DLPNO-CCSD(T)-F12D cc-pvtz-f12 def2/J ...,0,2,-522.306628,3727.0,"[[-2.125382, 2.640735, 1.073058], [-0.895562, ...",1.11863,0.011383
1,[C:1](=[C:2]([C:3]([C:4]([H:15])([H:16])[H:17]...,uHF UNO DLPNO-CCSD(T)-F12D cc-pvtz-f12 def2/J ...,0,2,-407.865447,2192.0,"[[3.393685, -0.431134, -0.110467], [2.143888, ...",0.60224,0.011839


Contains 340698 rows


In [13]:
rows_to_delete = set()
for col in dlpno_df.columns:
    to_delete = dlpno_df.index[dlpno_df[col].isna()]
    rows_to_delete.update(to_delete)
    if len(to_delete) > 0:
        print(f"Column {col} has {len(to_delete)} missing values")

print(f"\nMarked {len(rows_to_delete)} rows for deletion")

Column energy has 1335 missing values
Column run_time has 1335 missing values
Column dipole_au has 1335 missing values
Column t1_diagnostic has 931 missing values

Marked 1335 rows for deletion


In [14]:
dlpno_sp_from_species_data = dlpno_df["smiles"].map(
    species_data.set_index("asmi")["species_dlpno_sp_hartree"]
)
to_delete = dlpno_df.index[dlpno_df["energy"] != dlpno_sp_from_species_data]
rows_to_delete.update(to_delete)

print(f"\nA total of {len(rows_to_delete)} rows are now marked for deletion.")


A total of 1335 rows are now marked for deletion.


In [15]:
keep_index = np.ones(len(dlpno_df), dtype=bool)
keep_index[np.array(list(rows_to_delete), dtype=int)] = False
filtered_dlpno_data = dlpno_data.filter(pa.array(keep_index))
del dlpno_data

In [16]:
data_dir.mkdir(parents=True, exist_ok=True)
pq.write_table(filtered_dlpno_data, data_dir / output_file, compression="zstd")

In [17]:
parquet_file = pq.ParquetFile(data_dir / output_file)
print(f"Successfully wrote parquet file with {parquet_file.metadata.num_rows} rows")

Successfully wrote parquet file with 339363 rows
